# Forecast Walkthrough

This notebook is the presentation layer for the current Barents lice forecasting case.

It loads saved artifacts by default, so it opens quickly and stays reproducible. The notebook now highlights three upgrades beyond the first baseline:
- lagged neighbor-pressure features based on nearby sites,
- selective GPU XGBoost where it actually improves holdout results,
- an interactive map viewer built on top of the exported site snapshot.
        


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT
        


In [ ]:
import json
import numpy as np
import pandas as pd

from lice.config import PROCESSED_DIR, RESULTS_DIR

RUN_PIPELINE = False
if RUN_PIPELINE:
    from lice.pipeline import main
    main()

artifact_paths = {
    'audit': RESULTS_DIR / 'data_audit.json',
    'metrics': RESULTS_DIR / 'model_metrics.json',
    'comparison': RESULTS_DIR / 'model_comparison.csv',
    'errors': RESULTS_DIR / 'error_summary_by_area.csv',
    'predictions': RESULTS_DIR / 'holdout_predictions.csv',
    'benchmark': RESULTS_DIR / 'xgb_short_horizon_benchmark.csv',
    'map': RESULTS_DIR / 'site_map.geojson',
    'master': PROCESSED_DIR / 'master_table.parquet',
}
artifact_paths
        


In [ ]:
with open(artifact_paths['audit'], 'r', encoding='utf-8') as handle:
    audit_summary = json.load(handle)

with open(artifact_paths['metrics'], 'r', encoding='utf-8') as handle:
    metrics = json.load(handle)

comparison = pd.read_csv(artifact_paths['comparison'])
errors = pd.read_csv(artifact_paths['errors'])
predictions = pd.read_csv(artifact_paths['predictions'], parse_dates=['week_start_date'])
benchmark = pd.read_csv(artifact_paths['benchmark'])
map_data = json.loads(artifact_paths['map'].read_text(encoding='utf-8'))
map_sites = pd.DataFrame([feature['properties'] for feature in map_data['features']])
master_preview = pd.read_parquet(
    artifact_paths['master'],
    columns=[
        'sitenumber',
        'sitename',
        'week_start_date',
        'productionarea',
        'femaleadult',
        'mobilelice',
        'seatemperature',
        'treatment_count',
        'neighbor_breach_this_week_lag1',
        'neighbor_femaleadult_to_limit_ratio_lag1',
    ],
).sort_values(['week_start_date', 'sitenumber'], ascending=[False, True]).head(20)

audit_summary
        


In [ ]:
metrics_rows = []
for horizon, horizon_metrics in metrics.items():
    for model_type, values in horizon_metrics.items():
        row = {'horizon': horizon, 'model_type': model_type}
        row.update(values)
        metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows).sort_values(['horizon', 'model_type']).reset_index(drop=True)
selected_models = metrics_df[
    [
        'horizon',
        'model_type',
        'selected_model',
        'f1',
        'pr_auc',
        'rmse',
        'mae',
        'holdout_start',
        'holdout_end',
    ]
]
selected_models
        


In [ ]:
selected_classifier = metrics_df[metrics_df['model_type'] == 'classifier_any'][
    ['horizon', 'selected_model', 'f1', 'pr_auc']
].copy()
selected_classifier['horizon_weeks'] = selected_classifier['horizon'].str.replace('w', '').astype(int)

selected_regressor = metrics_df[metrics_df['model_type'] == 'regressor_count'][
    ['horizon', 'selected_model', 'rmse', 'mae']
].copy()
selected_regressor['horizon_weeks'] = selected_regressor['horizon'].str.replace('w', '').astype(int)

classifier_policy = benchmark[benchmark['task_type'] == 'classifier_any'][
    ['horizon', 'candidate_model', 'holdout_f1', 'holdout_pr_auc']
].merge(
    selected_classifier,
    left_on='horizon',
    right_on='horizon_weeks',
    how='left',
)
classifier_policy['task_type'] = 'classifier_any'
classifier_policy['decision'] = np.where(
    classifier_policy['selected_model'] == classifier_policy['candidate_model'],
    'adopted',
    'kept hist gradient boosting',
)
classifier_policy = classifier_policy[
    [
        'horizon',
        'task_type',
        'candidate_model',
        'holdout_f1',
        'holdout_pr_auc',
        'selected_model',
        'f1',
        'pr_auc',
        'decision',
    ]
]

regressor_policy = benchmark[benchmark['task_type'] == 'regressor_count'][
    ['horizon', 'candidate_model', 'holdout_rmse', 'holdout_mae']
].merge(
    selected_regressor,
    left_on='horizon',
    right_on='horizon_weeks',
    how='left',
)
regressor_policy['task_type'] = 'regressor_count'
regressor_policy['decision'] = np.where(
    regressor_policy['selected_model'] == regressor_policy['candidate_model'],
    'adopted',
    'kept hist gradient boosting',
)
regressor_policy = regressor_policy[
    [
        'horizon',
        'task_type',
        'candidate_model',
        'holdout_rmse',
        'holdout_mae',
        'selected_model',
        'rmse',
        'mae',
        'decision',
    ]
]

policy_summary = pd.concat([classifier_policy, regressor_policy], ignore_index=True)
policy_summary
        


In [ ]:
pd.DataFrame([audit_summary]).T.rename(columns={0: 'value'})
        


In [ ]:
master_preview
        


In [ ]:
comparison
        


In [ ]:
map_sites.sort_values(['max_breach_risk', 'femaleadult_to_limit_ratio'], ascending=[False, False])[
    [
        'sitename',
        'productionarea',
        'latest_observation_date',
        'risk_band',
        'classifier_1w_score',
        'classifier_2w_score',
        'classifier_12w_score',
        'count_1w_prediction',
        'count_2w_prediction',
        'count_12w_prediction',
        'last_treatment_date',
        'last_treatment_action',
        'last_treatment_activeingredient',
    ]
].head(20)
        


In [ ]:
predictions[
    (predictions['model_type'] == 'classifier_any') & (predictions['horizon'] == 12)
].sort_values('score', ascending=False).head(25)
        


In [ ]:
errors.groupby(['horizon', 'error_type']).head(10).reset_index(drop=True)
        


## Interactive map

The static viewer reads `results/site_map.geojson` and renders it with MapLibre.

Serve the repository root and open the viewer in a browser:

```powershell
python -m http.server 8765
```

Then browse to `http://127.0.0.1:8765/viewer/`.

Useful details in the viewer:
- current site status from the latest reported row,
- last treatment action and active ingredient,
- lagged production-area and neighbor-pressure context,
- latest validated holdout predictions for 1w, 2w, and 12w.
        


## Interview flow

Suggested sequence for the walkthrough:
1. Show the audit table and explain the site-week grain.
2. Show the selected-model table so the horizon-specific policy is clear.
3. Show the short-horizon benchmark summary to explain why XGBoost was kept only where it improved holdout performance.
4. Show the master-table preview to make the neighbor-pressure feature slice concrete.
5. Open the interactive map viewer to inspect high-risk sites, current status, treatment context, and validated forecast history.
6. Show the top 12-week predictions and the area-level error summary as the honest closing section.
        
